[Open in Colab](https://colab.research.google.com/github/BoydVonstocker/Introduction-to-Quantum-Computing/blob/main/Notebks/06%20-%20Deutsch_Algorithm.ipynb)


# Notebook 06 — Deutsch's Algorithm

Deutsch's problem is the smallest example of a quantum speed-up that is easy to hold in your head.

**Problem.** You are given an oracle for a function

$$
f:\{0,1\}\to\{0,1\}
$$

which is promised to be either **constant** ($f(0)=f(1)$) or **balanced** ($f(0)\neq f(1)$). Classically you must evaluate $f$ twice. Deutsch's algorithm decides the promise with **one** coherent query.


In [ ]:
!pip install qiskit qiskit-aer pylatexenc -q


In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram, array_to_latex
from qiskit_aer import AerSimulator


## The four possible functions

| name | $f(0)$ | $f(1)$ | type |
|---|---|---|---|
| `const0` | 0 | 0 | constant |
| `const1` | 1 | 1 | constant |
| `identity` | 0 | 1 | balanced |
| `NOT` | 1 | 0 | balanced |

The two-qubit *bit oracle* implements

$$
U_f\,|x\rangle|y\rangle = |x\rangle|y\oplus f(x)\rangle.
$$

If the target qubit starts in $|-\rangle = HX|0\rangle$, this becomes a *phase oracle* on the data qubit:

$$
U_f\,|x\rangle|-\rangle = (-1)^{f(x)}\,|x\rangle|-\rangle.
$$


## Build the four bit oracles


In [ ]:
def bit_oracle(kind: str) -> QuantumCircuit:
    """Two-qubit oracle: |x>|y> -> |x>|y XOR f(x)>."""
    o = QuantumCircuit(2, name=f"Uf[{kind}]")
    if kind == "const0":
        pass
    elif kind == "const1":
        o.x(1)
    elif kind == "identity":
        o.cx(0, 1)
    elif kind == "NOT":
        o.x(0)
        o.cx(0, 1)
        o.x(0)
    else:
        raise ValueError(kind)
    return o


for kind in ("const0", "const1", "identity", "NOT"):
    print(kind)
    print(bit_oracle(kind).draw("text"))


## The Deutsch circuit

1. Prepare the data qubit in $|0\rangle$ and the target in $|1\rangle$.
2. Apply $H$ to both (data becomes $|+\rangle$, target becomes $|-\rangle$).
3. Query $U_f$ once.
4. Apply $H$ to the data qubit and measure it.

- measurement `0` means **constant**
- measurement `1` means **balanced**

Why it works: after the oracle the data qubit is $|+\rangle$ if $f(0)=f(1)$ and $|-\rangle$ if $f(0)\neq f(1)$. The final Hadamard maps those two states onto $|0\rangle$ and $|1\rangle$.


In [ ]:
def deutsch(kind: str) -> QuantumCircuit:
    qc = QuantumCircuit(2, 1)
    qc.x(1)
    qc.h(0)
    qc.h(1)
    qc.barrier(label="oracle")
    qc.compose(bit_oracle(kind), inplace=True)
    qc.barrier()
    qc.h(0)
    qc.measure(0, 0)
    return qc


qc = deutsch("identity")
qc.draw("mpl")


## Run every oracle


In [ ]:
sim = AerSimulator()
for kind in ("const0", "const1", "identity", "NOT"):
    qc = deutsch(kind)
    counts = sim.run(qc, shots=1024).result().get_counts()
    print(f"{kind:10s}  counts={counts}")


Constant oracles should return only `0`. Balanced oracles should return only `1`. That is the whole algorithm: one query, a definite answer (on an ideal simulator).


In [ ]:
qc = deutsch("const1")
display(qc.draw("mpl"))
plot_histogram(sim.run(qc, shots=1024).result().get_counts())


## Watch the state just before measurement

Drop the final measure and print the statevector. The data qubit (little-endian: the right-hand amplitude pair grouping) is $|0\rangle$ or $|1\rangle$ with certainty, tensored with $|-\rangle$ on the target.


In [ ]:
for kind in ("const0", "identity"):
    qc = QuantumCircuit(2)
    qc.x(1)
    qc.h([0, 1])
    qc.compose(bit_oracle(kind), inplace=True)
    qc.h(0)
    sv = Statevector.from_instruction(qc)
    print(kind)
    display(array_to_latex(sv, prefix=rf"$|\psi_{{{kind}}}\rangle = $"))


## Deutsch–Jozsa in one paragraph

Replace the single data qubit by $n$ qubits, each prepared with $H^{\otimes n}$. The promise is now: $f:\{0,1\}^n\to\{0,1\}$ is either constant or balanced (exactly half of the inputs return 1). One query still decides the promise. Measuring all zeros on the data register means constant; any other bitstring means balanced.

Deutsch is the $n=1$ case. You now have the pattern used by Bernstein–Vazirani and Simon as well: layer of $H$, one oracle, layer of $H$, measure.


## Exercises

1. Confirm by hand that $U_f|x\rangle|-\rangle = (-1)^{f(x)}|x\rangle|-\rangle$ for `identity`.
2. What happens if you forget to prepare the target in $|-\rangle$ and leave it in $|0\rangle$?
3. Implement Deutsch–Jozsa for $n=2$ with a balanced oracle $f(x_0,x_1)=x_0$ (a single CNOT from data qubit 0 onto the target).


In [ ]:
# Exercise workspace
